In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## Data preparation

* Download the dataset
* Load it with pandas
* Make the column names and the string values consistent

In [ ]:
data = ""               #inserire link da kaggle
!wget $data

Parlare di ciò che è contenuto nelle colonne e di quello che si vuole predire.

In [ ]:
df = pd.read_csv("data.csv")
df.head().T

Looking at the column names, there is some inconsistency. 

Sometimes there are uppercase letters (Make, Model), sometimes there are spaces (Engine Fuel Type), sometimes underscores (Driven_Wheels).

* We make everything lowercase and replace spaces with underscores.

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

Per trovare le colonne contenenti stringhe e normalizzare i valori di tali colonne:

In [ ]:
df.dtypes

In [ ]:
strings = list(df.dtypes[df.dtypes == 'str'].index)

In [ ]:
for col in strings:
    df[col] = df[col].str.lower().str.replace(' ', '_')

In [ ]:
df.head().T               #check per vedere se gli elementi sono a posto

## Exploratory data analysis

In [ ]:
for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print()

In [ ]:
sns.histplot(df.msrp, bins = 50)                            #per vedere histogramma di una particolare colonna
sns.histplot(df.msrp[df.msrp < 100000], bins = 50)          #per focalizzarci su un particolare intervallo di quella colonna

price_logs = np.log1p(df.msrp)                              #applichiamo il logaritmo per comprimere i valori grandi, nal caso di long tail distribution
sns.histplot(price_logs, bins = 50)

In [ ]:
df.isnull().sum()                                           #per controllare se ci sono valori nulli e quanti per ogni colonna

## Validation framework

We take our dataset and split it into three parts:
* The first part we use for training, the second one for validation, and the last one for testing.
* We train a model, check if it works fine on the validation dataset, and leave the test dataset alone: we use it only very occasionally, at the very end, to check that our model is doing good.
* We will do a 60/20/20 split.

In [ ]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

In [ ]:
np.random.seed(2)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train+n_val]]
df_test = df.iloc[idx[n_train+n_val:]]

In [ ]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

The dataframes will be used to make the feature matrix X. The target *y* comes from the "msrp" column, and we already know that we need to apply the log1p transformation to it because of the long tail in its distribution. 
* Instead of keeping a pandas series, we immediately take its values with .values to get a NumPy array - we don't need the indexes and other pandas things for y:

In [ ]:
y_train = np.log1p(df_train.msrp.values)
y_val = np.log1p(df_val.msrp.values)
y_test = np.log1p(df_test.msrp.values)

After splitting the data I always take the target out into a separate variable and delete it from the dataframe completely:

In [ ]:
del df_train['msrp']
del df_val['msrp']
del df_test['msrp']

* Faccio selezione delle colonne che mi interessano (dividi numerical da categorical); nel vaso di linear regression usa solo numerical

## Linear regression

In [ ]:
def prepare_X_linear_regression(df):
    df = df.copy()
    df_num = df[numerical].fillna(0)
    X = df_num.values
    return X

In [ ]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [ ]:
def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [ ]:
X_train = prepare_X_linear_regression(df_train)
w0, w = train_linear_regression(X_train, y_train)

In [ ]:
X_val = prepare_X_linear_regression(df_val)
y_pred = w0 + X_val.dot(w)

In [ ]:
rmse(y_val, y_pred)                                     #root mean square error

In [ ]:
sns.histplot(y_pred, color='red', alpha=0.5, bins=50)
sns.histplot(y_val, color='blue', alpha=0.5, bins=50)

Si potrebbe migliorare il valore di rmse aggiungendo una colonna con dei nuovi valori:

In [ ]:
def prepare_X_linear_regression(df):
    df = df.copy()
    features = numerical.copy()

    df['age'] = 2017 - df.year      #colonna nuova
    features.append('age')

    for v in [2, 3, 4]:                                                         #categorical variables
        df['num_doors_%s' % v] = (df.number_of_doors == v).astype('int')
        features.append('num_doors_%s' % v)

    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X